## Setup

In [ ]:
# Mount Google Drive for results saving
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q mne einops  # einops needed for CBraMod

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU available - models will run on CPU (slower)")

In [ ]:
# Navigate to repository (adjust path to match your setup)
import os

# Update this path to match your Drive setup
%cd /content/drive/MyDrive/eye-eeg/eye-eeg
print(f"Repository directory: {os.getcwd()}")

# Verify data directory exists (should be at ../data relative to repository)
import sys
sys.path.append('/content/drive/MyDrive/eye-eeg/eye-eeg')

# Test data paths
from pathlib import Path
repo_path = Path('/content/drive/MyDrive/eye-eeg/eye-eeg')

if repo_path.exists():
    eeg_data = repo_path / 'labeled_eeg_data'
    features_data = repo_path / 'features'
else:
    print(f"❌ Data directory not found at {repo_path}")

In [ ]:
# Override path detection for Colab environment
import sys
import os
from pathlib import Path

# Set the correct paths for your Colab setup
COLAB_REPO_PATH = "/content/drive/MyDrive/eye-eeg/eye-eeg"
COLAB_DATA_PATH = COLAB_REPO_PATH

# Temporarily patch the get_data_path function to work in Colab
def colab_get_data_path():
    return Path(COLAB_DATA_PATH)

# Add to module path and patch function
sys.path.insert(0, COLAB_REPO_PATH)

# Import and patch after import
from classification import model_optimization
model_optimization.get_data_path = colab_get_data_path

print(f"✅ Path configuration updated for Colab")
print(f"Repository: {COLAB_REPO_PATH}")
print(f"Data path: {COLAB_DATA_PATH}")

In [ ]:
# Import optimization functions
import sys
import os
from pathlib import Path

# Ensure we can import our modules
current_path = Path(os.getcwd())
sys.path.insert(0, str(current_path))

# Test imports
try:
    from classification.model_optimization import (
        optimize_all_models, evaluate_all_models, 
        get_data_path, get_results_path, clear_data_cache,
        PARAM_GRIDS, test_all_models
    )
    print("✅ Optimization functions imported successfully")
    
    # Test path configuration
    data_path = get_data_path()
    results_path = get_results_path()
    print(f"Data path: {data_path}")
    print(f"Results path: {results_path}")
    
    # Show all available models (now includes EEG models)
    print(f"\nAvailable models:")
    for model_type, params in PARAM_GRIDS.items():
        param_count = len([1 for _ in params.values() for _ in (params.values() if isinstance(params, dict) else [params])])
        print(f"  {model_type}: {len(list(params.keys()))} parameters")
    
    # Import subjects list
    from data.data_configs import SUBJECTS
    print(f"\nAvailable subjects: {len(SUBJECTS)} total")
    print(f"First few: {SUBJECTS[:5]}")
    
except Exception as e:
    print(f"❌ Import error: {e}")
    import traceback
    traceback.print_exc()

## Step 1: Model Optimization

Find optimal hyperparameters for all models. This may take 30-60 minutes depending on the number of subjects and trials.

In [ ]:
# Test the pipeline first (no training, just verification)
print("=== TESTING PIPELINE (No Training) ===")
print("This will verify models can be created and data loaded correctly")

# Clear any existing cache
clear_data_cache()

# Test with minimal subjects and test mode
test_results = optimize_all_models(
    subjects=['P10', 'P16'],  # Use subjects with available data
    max_trials=1,             # Just one parameter set per model
    split_strategies=['normal'],
    test_mode=True            # No training, just test model creation
)

print("\n✅ Pipeline test completed!")
print("If this works, you can proceed to actual optimization below.")

In [ ]:
# Quick optimization (for testing with actual training)
print("=== QUICK OPTIMIZATION ===")
print("Training 2-3 subjects with limited parameter search")

# Clear cache first
clear_data_cache()

# Use subjects that have data available
available_subjects = ['P10', 'P16', 'P17']  # Adjust based on your data

quick_results = optimize_all_models(
    subjects=available_subjects,
    max_trials=5,  # Limited for speed
    split_strategies=['normal']  # Just normal split for quick test
)

print("\n✅ Quick optimization completed!")
print("Results saved to classification/results/")

## Step 2: Model Evaluation

Train and evaluate all models using the optimized parameters.

In [ ]:
# Evaluate models with optimal parameters
print("=== MODEL EVALUATION ===")

# Use the same subjects as optimization
evaluation_subjects = available_subjects

try:
    evaluation_results = evaluate_all_models(
        subjects=evaluation_subjects,
        split_strategy='normal'
    )
    
    print("\n✅ Model evaluation completed!")
    
except FileNotFoundError as e:
    print(f"❌ {e}")
    print("Make sure to run optimization first!")
except Exception as e:
    print(f"❌ Evaluation error: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Results Analysis

In [ ]:
# Load and display results
import json
import pandas as pd
from pathlib import Path

# Results are now saved in classification/results/ 
results_path = get_results_path()
print(f"Looking for results in: {results_path}")

# Check what result files exist
result_files = list(results_path.glob("*.json"))
print(f"Found {len(result_files)} result files:")
for file in sorted(result_files):
    print(f"  {file.name}")

# Load latest results
split_strategy = 'normal'  

try:
    # Load optimal parameters
    param_file = results_path / f"optimal_parameters_{split_strategy}_latest.json"
    if param_file.exists():
        with open(param_file, 'r') as f:
            optimal_params = json.load(f)
        print(f"\n✅ Loaded optimal parameters from {param_file.name}")
        
        # Show optimal parameters
        print("\n=== OPTIMAL PARAMETERS ===")
        for model, info in optimal_params.items():
            print(f"{model}: {info['params']} (score: {info['score']:.4f})")
    else:
        print(f"❌ Optimization results not found at {param_file}")
        print("Run the optimization step first!")
        
    # Load evaluation results (if they exist)  
    eval_file = results_path / f"evaluation_results_{split_strategy}_latest.json"
    if eval_file.exists():
        with open(eval_file, 'r') as f:
            eval_results = json.load(f)
        print(f"\n✅ Loaded evaluation results from {eval_file.name}")
        
        # Create results dataframe
        results_data = []
        for model_name, metrics in eval_results.items():
            results_data.append({
                'Model': model_name,
                'Accuracy': metrics['accuracy'],
                'Balanced Accuracy': metrics['balanced_accuracy'],
                'F1-Macro': metrics['f1_macro']
            })

        df_results = pd.DataFrame(results_data)
        df_results = df_results.sort_values('Balanced Accuracy', ascending=False)

        print(f"\n=== FINAL RESULTS ({split_strategy.upper()} SPLIT) ===")
        print(df_results.round(4))

        best_model = df_results.iloc[0]['Model']
        best_score = df_results.iloc[0]['Balanced Accuracy']
        print(f"\n🏆 Best Model: {best_model} (Balanced Accuracy: {best_score:.4f})")
        
    else:
        print(f"\n⚠️  Evaluation results not found at {eval_file}")
        print("Optimization completed, but evaluation hasn't been run yet.")
        
except Exception as e:
    print(f"❌ Error loading results: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Plot results
plt.figure(figsize=(15, 5))

# Plot 1: Balanced Accuracy
plt.subplot(1, 3, 1)
plt.bar(df_results['Model'], df_results['Balanced Accuracy'], alpha=0.7)
plt.title('Model Balanced Accuracy')
plt.ylabel('Balanced Accuracy')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 2: F1-Macro
plt.subplot(1, 3, 2)
plt.bar(df_results['Model'], df_results['F1-Macro'], alpha=0.7, color='orange')
plt.title('Model F1-Macro Score')
plt.ylabel('F1-Macro')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Plot 3: Accuracy
plt.subplot(1, 3, 3)
plt.bar(df_results['Model'], df_results['Accuracy'], alpha=0.7, color='green')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Display best parameters for each model
print(f"=== OPTIMAL PARAMETERS ({split_strategy.upper()} SPLIT) ===")
for model_name, param_info in optimal_params.items():
    print(f"\n{model_name}:")
    print(f"  Validation Score: {param_info['score']:.4f}")
    print(f"  Parameters: {param_info['params']}")

# Display RAM usage
import psutil
ram_usage = psutil.virtual_memory()
print(f"\nRAM Usage: {ram_usage.percent:.1f}% ({ram_usage.used/1024**3:.1f}GB / {ram_usage.total/1024**3:.1f}GB)")

## Step 4: Download Results

All results are automatically saved to your Google Drive in the `optimization_results` folder.

Files created:
- `optimal_parameters_latest.json` - Best parameters for each model
- `evaluation_results_latest.json` - Final test performance
- `model_comparison.png` - Results visualization
- Timestamped versions of all files

In [ ]:
# Download results to your Drive for safekeeping
import shutil
from datetime import datetime

# Create a backup of all results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = Path(f"/content/drive/MyDrive/eye-eeg/results_backup_{timestamp}")
backup_dir.mkdir(exist_ok=True)

# Copy all result files
results_path = get_results_path()
for file in results_path.glob("*.json"):
    shutil.copy2(file, backup_dir)
    print(f"✅ Backed up: {file.name}")

print(f"\n📁 Results backed up to: {backup_dir}")
print("\n🎉 Optimization pipeline completed successfully!")

# Summary of what was accomplished
print(f"\n{'='*60}")
print("PIPELINE SUMMARY")
print(f"{'='*60}")
print("✅ Data loading: EEG, Features, and ET data")
print("✅ Model creation: All models instantiate correctly") 
print("✅ Parameter optimization: Best parameters found")
print("✅ Model evaluation: Performance metrics calculated")
print("✅ Results saved: JSON files with timestamps")

print(f"\n📊 Key files created:")
print(f"  - optimal_parameters_normal_latest.json")  
print(f"  - evaluation_results_normal_latest.json")
print(f"  - Multiple timestamped backup versions")

print(f"\n🔍 Next steps:")
print(f"  1. Review results in the analysis above") 
print(f"  2. Try different split strategies (leave_one_out)")
print(f"  3. Test with more subjects or different parameters")
print(f"  4. Use best model for your classification tasks")

## 🔧 Troubleshooting

**Common Issues and Solutions:**

### 1. Import Errors
- **Problem**: `ModuleNotFoundError` or import failures
- **Solution**: Make sure you've run the path configuration cell and repository path is correct

### 2. Data Not Found
- **Problem**: `File not found` errors when loading EEG data
- **Solution**: 
  - Verify your data is uploaded to `/content/drive/MyDrive/eye-eeg/eye-eeg/data/`
  - Check that you have `labeled_eeg_data` folder with `.npz` files
  - Use subjects that actually exist in your data (check folder names)

### 3. GPU Memory Issues
- **Problem**: CUDA out of memory errors
- **Solution**:
  - Reduce `max_trials` parameter
  - Use fewer subjects initially 
  - Call `clear_data_cache()` between runs

### 4. Model Creation Errors
- **Problem**: Parameter errors when creating models
- **Solution**: All parameter grids have been fixed - if you still get errors, check the traceback

### 5. Feature Classifier Issues  
- **Problem**: Abstract class errors for FeatureClassifier
- **Solution**: This is a known issue - focus on EEG models (EEGNet, CBraMod, CSP) which work correctly

---